# 📊 Data Cleaning & Visualization Project
### Thiranex Internship | Task 1
**Intern:** Suraj | **ID:** THX-JUN0526-1465 | **Role:** Data Science Intern

---

## STEP 0: Install & Import Libraries

In [ ]:
# Install required libraries (only needed in Colab)
!pip install pandas matplotlib seaborn -q

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')

## STEP 1: Load the Dataset

In [ ]:
# Load dataset directly from GitHub
url = 'https://raw.githubusercontent.com/barcaboi17/thiranex-internship-task1/main/sales_data.csv'
df = pd.read_csv(url)

print(f'✅ Dataset loaded! Rows: {df.shape[0]} | Columns: {df.shape[1]}')
df.head()

## STEP 2: Explore the Data

In [ ]:
print('📋 Data Types:')
print(df.dtypes)
print()
print('📊 Basic Statistics:')
df.describe()

In [ ]:
print('❓ Missing Values:')
print(df.isnull().sum())
print(f'\n🔁 Duplicate Rows: {df.duplicated().sum()}')

## STEP 3: Data Cleaning

In [ ]:
# 3a. Remove Duplicates
before = len(df)
df = df.drop_duplicates()
print(f'[1] Removed {before - len(df)} duplicate row(s). Rows remaining: {len(df)}')

In [ ]:
# 3b. Handle Missing Values
median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)
print(f'✅ Age missing values filled with median: {median_age}')

df['Gender'] = df['Gender'].fillna('Unknown')
print('✅ Gender missing values filled with Unknown')

mean_rating = round(df['Rating'].mean(), 2)
df['Rating'] = df['Rating'].fillna(mean_rating)
print(f'✅ Rating missing values filled with mean: {mean_rating}')

In [ ]:
# 3c. Handle Outliers
invalid_age = df[df['Age'] < 0].shape[0]
df = df[df['Age'] >= 0]
print(f'✅ Removed {invalid_age} row(s) with negative Age')

Q1 = df['Price'].quantile(0.25)
Q3 = df['Price'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR
outlier_count = df[df['Price'] > upper_limit].shape[0]
df = df[df['Price'] <= upper_limit]
print(f'✅ Removed {outlier_count} row(s) with extreme Price outliers (threshold: ₹{upper_limit:,.0f})')

In [ ]:
# 3d. Fix Data Types
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Age'] = df['Age'].astype(int)
df['Month'] = df['Order_Date'].dt.strftime('%b')
df['Month_Num'] = df['Order_Date'].dt.month

print(f'✅ Data Cleaning Complete! Clean dataset: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

## STEP 4: Visualizations — Sales Dashboard

In [ ]:
sns.set_style('whitegrid')
sns.set_palette('husl')

fig = plt.figure(figsize=(18, 14))
fig.suptitle('Thiranex Internship | Sales Data Dashboard\nIntern: Suraj | THX-JUN0526-1465',
             fontsize=16, fontweight='bold', y=0.98)

# Chart 1: Sales by Category
ax1 = fig.add_subplot(2, 3, 1)
category_sales = df.groupby('Category')['Total_Sales'].sum().sort_values(ascending=False)
bars = ax1.bar(category_sales.index, category_sales.values, color=['#2ecc71', '#3498db', '#e74c3c'])
ax1.set_title('💰 Total Sales by Category', fontweight='bold')
ax1.set_xlabel('Category')
ax1.set_ylabel('Total Sales (₹)')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
for bar, val in zip(bars, category_sales.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'₹{val/1000:.0f}K', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Chart 2: Gender Distribution
ax2 = fig.add_subplot(2, 3, 2)
gender_counts = df['Gender'].value_counts()
ax2.pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%',
        colors=['#ff6b9d', '#4ecdc4', '#95a5a6'], startangle=90, explode=[0.05]*len(gender_counts))
ax2.set_title('👥 Customer Gender Distribution', fontweight='bold')

# Chart 3: Monthly Sales Trend
ax3 = fig.add_subplot(2, 3, 3)
monthly = df.groupby(['Month_Num', 'Month'])['Total_Sales'].sum().reset_index().sort_values('Month_Num')
ax3.plot(monthly['Month'], monthly['Total_Sales'], marker='o', linewidth=2.5,
         color='#e67e22', markersize=8, markerfacecolor='white', markeredgewidth=2)
ax3.fill_between(monthly['Month'], monthly['Total_Sales'], alpha=0.1, color='#e67e22')
ax3.set_title('📈 Monthly Sales Trend', fontweight='bold')
ax3.set_xlabel('Month')
ax3.set_ylabel('Total Sales (₹)')
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))

# Chart 4: Top 5 Cities
ax4 = fig.add_subplot(2, 3, 4)
city_sales = df.groupby('City')['Total_Sales'].sum().nlargest(5)
bars4 = ax4.barh(city_sales.index, city_sales.values, color=sns.color_palette('viridis', len(city_sales)))
ax4.set_title('🏙️ Top 5 Cities by Sales', fontweight='bold')
ax4.set_xlabel('Total Sales (₹)')
ax4.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
for bar, val in zip(bars4, city_sales.values):
    ax4.text(val + 200, bar.get_y() + bar.get_height()/2, f'₹{val/1000:.0f}K', va='center', fontsize=9)

# Chart 5: Rating Distribution
ax5 = fig.add_subplot(2, 3, 5)
ax5.hist(df['Rating'], bins=10, color='#9b59b6', edgecolor='white', linewidth=0.8)
ax5.set_title('⭐ Customer Rating Distribution', fontweight='bold')
ax5.set_xlabel('Rating')
ax5.set_ylabel('Number of Orders')
ax5.axvline(df['Rating'].mean(), color='red', linestyle='--',
            linewidth=1.5, label=f"Mean: {df['Rating'].mean():.2f}")
ax5.legend()

# Chart 6: Box Plot
ax6 = fig.add_subplot(2, 3, 6)
categories = df['Category'].unique()
data_by_category = [df[df['Category'] == cat]['Rating'].values for cat in categories]
bp = ax6.boxplot(data_by_category, labels=categories, patch_artist=True)
for patch, color in zip(bp['boxes'], ['#2ecc71', '#3498db', '#e74c3c']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax6.set_title('📦 Rating by Product Category', fontweight='bold')
ax6.set_xlabel('Category')
ax6.set_ylabel('Rating')

plt.tight_layout()
plt.savefig('sales_dashboard.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('✅ Dashboard saved!')

## STEP 5: Key Insights

In [ ]:
print('💡 KEY INSIGHTS')
print('=' * 45)
print(f'1. 🏆 Top Sales Category : {category_sales.idxmax()}')
print(f'2. 🏙️  Best Performing City: {city_sales.idxmax()}')
print(f'3. 🛒 Best Selling Product: {df.groupby("Product")["Total_Sales"].sum().idxmax()}')
print(f'4. ⭐ Average Rating      : {df["Rating"].mean():.2f} / 5.0')
print(f'5. 📦 Total Orders        : {len(df)}')
print(f'6. 💰 Total Revenue       : ₹{df["Total_Sales"].sum():,.0f}')